# Operation Midnight Launch

**CAS Building ML/AI Applications, HS26. Weekend 1, Saturday parallel session.**

Time: about 35 minutes. Tools: VS Code, a terminal, Claude Code, git.
Output: one `index.html` and a git history somebody else can read.

---

## Read this first: where this notebook runs

**This notebook runs on your own machine, not in Google Colab.** Every other
coding exercise in this course is a Colab notebook, and this one is the
exception, for one reason: the thing you are practising is a program that
lives in your terminal and edits files in a folder you own. Colab has neither
your folder nor your Claude licence.

So the split is:

| Where | What happens there |
|---|---|
| **This notebook** | The mechanical setup. Making the folder, writing the starter files, running the git commands, showing you the history. Run the cells. |
| **Your terminal** | Claude Code itself. It is an interactive program, so it cannot run inside a notebook cell. Every prompt you have to type is written out for you to paste. |

Open the notebook with `jupyter lab`, or in VS Code, from any directory.
It creates the project folder for you.

## Before you start

You should already have:

- **VS Code** installed and opening correctly.
- **Claude Code** installed, with `claude --version` printing a version number,
  and signed in at least once.
- **git** installed, with `git --version` printing a version number.

The next cell checks all three and tells you what is missing.

**If anything is missing, or if you have never opened a terminal**, work
through the course setup guide first. It walks through installing VS Code,
opening a terminal, accepting your Claude invitation and installing Claude
Code, and it has a button that copies every command so nothing has to be typed
by hand:

https://eth-bmai-hs26.github.io/BMAI-PAGE/guides/claude-code-setup.html

In [ ]:
import shutil, subprocess, sys
from pathlib import Path

def version_of(tool, args=("--version",)):
    "Return the first line a tool prints for --version, or None if it is absent."
    exe = shutil.which(tool)
    if exe is None:
        return None
    try:
        out = subprocess.run([exe, *args], capture_output=True, text=True, timeout=30)
    except Exception as exc:
        return "found, but did not answer: %s" % exc
    return (out.stdout or out.stderr).strip().splitlines()[0]

print("python  ", sys.version.split()[0])
missing = []
for tool in ("git", "claude", "code"):
    v = version_of(tool)
    print("%-8s" % tool, v if v else "NOT FOUND")
    if v is None:
        missing.append(tool)

if "git" in missing or "claude" in missing:
    print("\nStop here and install what is missing before going on.")
elif "code" in missing:
    print("\nThe VS Code command line launcher is not on your PATH. That is fine:")
    print("open the folder from the VS Code menu instead, with File then Open Folder.")
else:
    print("\nAll three are here. Go on.")

---

## The story

It is 11 at night on a Thursday.

You have just taken a panicked call from **Mara**, the CEO of **NovaGrid**, a
clean energy analytics startup. Investors walk through the door at 9 in the
morning for a live demo of the company's new landing page.

Small problem. The freelance developer who was building it is gone. Vanished.
Phone off, profile deleted. What is left behind is a half finished
`index.html`, a cold cup of coffee, and a sticky note that reads:

> *"good luck"*

You have been promoted, voluntarily or otherwise, to **Interim Head of
Product**. Your mission is to ship this landing page before sunrise.

The good news: you have an engineer. A tireless, endlessly patient, slightly
literal engineer named **Claude Code**.

The bad news: the board cut the AI tooling budget last quarter, so you have to
be smart about which model you use and how much context you burn. And **Elena**
(CTO, currently asleep, terrifying when awake) will inspect your **git history**
first thing in the morning. She expects one clean commit per ticket, each with
a message that says what it was, and no mess.

No pressure.

## What you will practise

By the end of this exercise you will have hands on experience with:

- Setting up and prompting Claude Code.
- Writing a structured `CLAUDE.md` to steer your AI engineer.
- Using `/model`, `/clear`, `/compact` and `/context` deliberately.
- Saving one clean commit per ticket, and undoing a change that did not work.
- Pasting an image into the terminal as context.
- Writing your own slash command.
- Managing the context window, because tokens are the budget.

---

## If you have never opened a terminal

You will need one twice in this exercise: once to start Claude Code, and once
more if anything goes wrong. It is a window where you type one line and press
Enter.

**On a Mac.** Press `Cmd` and `Space` together, type `Terminal`, press `Enter`.

**On Windows.** Press `Win` and `X` together, then choose **Windows
PowerShell**, or **Terminal** if that is what the menu says.

**Inside VS Code**, which is where you will actually work: open your project
folder with **File**, then **Open Folder**, then open the panel with
**Terminal**, then **New Terminal**. It is the same program, docked at the
bottom of the editor.

Four things worth knowing before you start:

- You type one line and press `Enter`. Nothing happens until you press Enter.
- You cannot click on anything in there. Use the arrow keys.
- Text scrolling past is normal. It is the program saying what it is doing.
- Once Claude Code is running, `Esc` interrupts it and `exit` closes it.

The setup guide has the same steps with pictures, and a copy button on every
command: https://eth-bmai-hs26.github.io/BMAI-PAGE/guides/claude-code-setup.html

---

# Phase 0: the workspace (5 minutes)

## Step 1: choose where the project lives

The cell below puts `novagrid-demo` on your Desktop. Change `PROJECT_DIR` if you
would rather have it somewhere else, then run the cell.

In [ ]:
from pathlib import Path
import os, subprocess, textwrap

PROJECT_DIR = Path.home() / "Desktop" / "novagrid-demo"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)
print("Project folder:", PROJECT_DIR)


def run(cmd, cwd=PROJECT_DIR, quiet=False):
    "Run one shell command in the project folder and print what it said."
    done = subprocess.run(cmd, cwd=str(cwd), shell=True,
                          capture_output=True, text=True)
    if not quiet:
        print("$", cmd)
        for stream in (done.stdout, done.stderr):
            if stream.strip():
                print(stream.rstrip())
    return done.returncode

## Step 2: write the starter file

This is what the freelancer left behind. Yes, it is bad. That is the point.

The cell refuses to overwrite an `index.html` that is already there, so you can
re-run the notebook later without losing the work Claude did for you.

In [ ]:
STARTER_HTML = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>NovaGrid</title>
    <style>
        /* TODO: add styles I guess? */
        body {
            font-family: Arial, sans-serif;
            margin: 0;
            padding: 0;
        }
    </style>
</head>
<body>

    <header>
        <h1>NovaGrid</h1>
        <p>we do energy stuff</p>
        <!-- TODO: navigation maybe? -->
    </header>

    <main>
        <section>
            <h2>Welcome</h2>
            <p>Lorem ipsum dolor sit amet. This is placeholder text that the developer
            definitely meant to replace before disappearing forever.</p>
            <a href="#">Click here for something</a>
        </section>

        <section>
            <h2>Our Impact</h2>
            <p>We have saved lots of energy. Like, a LOT. Trust us.</p>
            <!-- TODO: some kind of numbers here? investors like numbers -->
        </section>
    </main>

    <footer>
        <p>NovaGrid 2026. All rights reserved probably.</p>
    </footer>

</body>
</html>
"""

index = PROJECT_DIR / "index.html"
if index.exists():
    print("index.html is already there, leaving it alone.")
    print("Delete it by hand if you want to start the exercise over.")
else:
    index.write_text(STARTER_HTML, encoding="utf-8")
    print("Wrote", index)

## Step 3: look at it

Run the cell to open the page in your browser. Keep that tab open. Every time
Claude changes something, refresh it.

In [ ]:
import webbrowser
webbrowser.open((PROJECT_DIR / "index.html").as_uri())
print("Opened", (PROJECT_DIR / "index.html").as_uri())

## Step 4: turn the folder into a repository

Elena will read these commit messages in the morning. She will understand.

In [ ]:
run("git init")
run("git add .")
run('git commit -m "chore: initial commit, inherited from missing developer"')

---

# Phase 1: the handover document (5 minutes)

Before you wake your AI engineer up, you write the handover document. The
freelancer did not leave one, so you write the one you wish they had.

**Why this matters.** Claude Code reads `CLAUDE.md` automatically when it
starts. Think of it as the onboarding document for your engineer. A good one
means fewer misunderstandings, less wasted context, and a calmer night for
everyone.

The next cell writes it. Read it before you run it: the seven rules in it are
the whole reason this exercise behaves the way it does, and later in the
evening you will be glad rule 5 is there.

In [ ]:
HANDOVER = """# NovaGrid: Investor Demo Page

## Your Role
You are a senior front-end engineer supporting a non-technical Head of Product
(that's me) overnight, before a critical investor demo at 9 AM.
You are calm, careful, and you've seen rushed launches go badly before.
You'd rather ship something solid than something flashy and broken.

## The Project
- Single static page: `index.html`
- Inline CSS and JS only (no external files, no frameworks, no npm, no build tools)
- The page must work by simply opening the file in a browser (double-click to open)
- Audience: investors seeing NovaGrid for the first time
- Tone: clean, confident, professional but not boring

## How You Work (Golden Rules)

### Rule 1: Orient Before You Act
Before making ANY change, briefly summarize:
  - What the relevant file currently looks like
  - What you understand the request to be
Confirm with me before writing code.

### Rule 2: One Change at a Time
Implement exactly what was asked. Nothing extra.
No "while I'm at it" additions. No unrequested libraries.
No surprise refactors. One ticket, one change.

### Rule 3: Explain in Plain Language
Before showing code, tell me in simple terms:
  - What you're about to change
  - Why this approach makes sense
I'm not a developer. Skip the jargon. If you must use a technical term,
explain it in the same sentence.

### Rule 4: Ask, Don't Guess
If a request is vague or ambiguous, ask ONE clarifying question
instead of making assumptions. Better to ask than to rebuild.

### Rule 5: Fail Gracefully
If something breaks:
  1. STOP immediately
  2. Tell me clearly what broke and why (in plain language)
  3. Recommend reverting to the last working state via git
  4. Do NOT attempt to patch on top of broken code
  5. Wait for my go-ahead before trying again
If a second attempt also fails, revert again and suggest
a simpler alternative approach.

### Rule 6: Hands Off
Never modify these files unless I explicitly ask:
  - `.git/` directory
  - `CLAUDE.md`
  - `.claude/` directory

### Rule 7: Commit Messages
Use short, imperative style with a prefix:
  - `feat: add hero section with CTA`
  - `fix: correct broken navigation link`
  - `chore: clean up unused CSS`

## Git Workflow
You are responsible for all git operations. I will never type one myself.
When I give you a task:
  1. Make the changes
  2. Wait for my approval before committing
  3. When I confirm, stage the changes and commit with a proper message
  4. One ticket, one commit. Never bundle two tickets into one commit
  5. Tell me, in one line, which git commands you ran

If I ask you to undo something, discard the uncommitted changes with
`git checkout -- .`. If it is already committed, undo that one commit with
`git reset --hard HEAD~1` and say plainly that you have done so.

## Communication Style
- Keep responses concise (I'm reading this at 2 AM)
- Use bullet points for lists, short paragraphs for explanations
- When reporting what you changed, give me a quick before/after summary
"""

handover = PROJECT_DIR / "CLAUDE.md"
if handover.exists():
    print("CLAUDE.md is already there, leaving it alone.")
else:
    handover.write_text(HANDOVER, encoding="utf-8")
    print("Wrote", handover)

In [ ]:
run("git add CLAUDE.md")
run('git commit -m "chore: add CLAUDE.md engineering handover"')

## Now wake the engineer up

Claude Code is interactive, so it cannot run inside a notebook cell. It runs in
a **terminal**. If you are not sure how to open one, the section above says how.

Run the cell below. It prints the exact two lines to paste into your terminal,
with your own project folder already filled in, so there is no path to type by
hand.

In [ ]:
# Print the two lines to paste into a terminal. Copy them from the output
# rather than typing them: a mistyped path is the commonest way this exercise
# stalls, and the folder is wherever PROJECT_DIR points, which you may have
# changed at the top.
print("Open a terminal, then paste these two lines, one at a time:")
print()
print('cd "%s"' % PROJECT_DIR)          # double quotes work in bash AND PowerShell
print("claude")
print()
print("The first line moves the terminal into your project folder.")
print("The second starts Claude Code there.")

Then say hello. As a warm up, type something like:

> Read through the project and tell me what we are working with. Do not change
> anything yet.

Claude should orient itself using `CLAUDE.md` and the existing `index.html`.
Read its summary carefully: this is your engineer telling you it understood the
handover.

**Tip.** Run `/model` and see which model you are on. For the first couple of
simple tasks, consider a lighter one. You can switch back for the hard parts.

---

# Phase 2: the tickets (15 minutes)

Tickets arrive from different NovaGrid stakeholders. For **each ticket** you
ask Claude Code to make the change, you look at it, and then you tell it to
save a snapshot. One ticket, one commit. You are the Head of Product, not the
engineer, so Claude does the git work too and you never type a git command.

**The cycle, once:**

1. **Give Claude the ticket**, in your own words.

2. **Review the result.** Refresh the browser tab. Does it look right? Does it
   match what the stakeholder asked for?

3. **If it is good, tell Claude to save it.**
   > Looks good. Save this as a snapshot with a message that says what it was.

4. **If it is wrong, tell Claude to undo it.**
   > This does not look right. Undo it and let us try a different approach.

   Remember rule 5 of your `CLAUDE.md`: do not pile fixes on top of broken
   code. Undo first, then re-prompt.

5. **Check your context.** Run `/context` after each ticket.

**Context rule of thumb.** After each ticket, run `/context`. If the window is
getting heavy and the next ticket is unrelated to what you just did, use
`/clear` and Claude re-reads `CLAUDE.md` fresh. If the next ticket builds on
this one, use `/compact` instead, which keeps the thread while freeing space.

### Ticket 1: the hero section

**From:** Mara (CEO). **Priority:** high.

> The hero section is embarrassing. We need something that actually tells
> people what NovaGrid does. We are a clean energy analytics platform that
> helps businesses track, reduce and report their carbon footprint. Make it
> inspiring. Add a call to action button that invites people to request a demo.
> I trust your taste, just make it look like we know what we are doing.

**Your move.** This is open ended on purpose. You decide how to turn Mara's
vision into a prompt. Think about which details would help Claude get it right
on the first try.

Example prompt to get you started:

> Redesign the hero section of index.html so that [your creative direction,
> based on Mara's message].

Once you are happy:

> Save this as a snapshot, with a message that says what it was.

Then run `/context`.

### Ticket 2: brand identity

**From:** Theo (Head of Design). **Priority:** high.

> Here are our brand guidelines. Please match these.

Theo did not write a spec. He sent a screenshot. Typical designer move.

**Your move.**

1. Find an image that could serve as NovaGrid's brand reference: a colour
   palette, a logo, any visual that feels like a clean energy startup.
2. **Paste or drag that image straight into the Claude Code terminal.**
3. Prompt Claude:

   > Here is our brand reference [paste your image]. Update the styling of
   > index.html to match these colours and this visual direction.

Once you are happy: *Save this as a snapshot.*

This is your chance to practise giving Claude visual context.

Then run `/context` again. If it is getting heavy, this is a natural moment for
`/clear`: the next ticket has nothing to do with styling.

### Ticket 3: legal compliance

**From:** Priya (Legal). **Priority:** medium.

> We need a proper footer. Include a copyright notice for the current year, a
> link to a privacy policy page (just use # as the href for now), and a short
> cookie disclaimer. Standard language is fine. GDPR also requires a "Manage
> preferences" link. Keep it professional.

**Your move.** This is a straightforward text task. Low complexity, low risk.

**Budget moment.** A good time to switch to a cheaper model. Run `/model` and
pick something light. A footer does not need the most capable engine.

> [Paste Priya's request.] Save it as a snapshot when you are done.

Since this is simple and well defined, you can let Claude do the whole cycle in
one go. Afterwards, switch the model back up. The next ticket needs it.

### Ticket 4: the wow factor

**From:** Sam (Investor Liaison). **Priority:** HIGH, all caps, his choice.

> The investors LOVE numbers. Can we add a section that shows NovaGrid's impact
> in a visually impressive way? Think big counters or stats, something that
> catches the eye when they scroll down. Tons of CO2 offset, renewable energy
> monitored, companies onboarded. Make it feel alive. Use your judgment on the
> design.

**Your move.** This is deliberately vague. Sam wants "wow" and has not defined
it. So either ask Claude a clarifying question, which is what rule 4 of your
`CLAUDE.md` invites, or supply your own creative direction in the prompt.

> [Your reading of Sam's request, with your own direction.]

**Heads up.** If Claude reaches for something over ambitious, an external
library or an animation framework, remember rule 2: one change at a time, no
unrequested extras. If the page breaks, do not panic and do not pile more
prompts on top:

> This broke the page. Undo everything you just did, then let us try a simpler
> approach: [your new direction].

Afterwards run `/context`. Four tickets in, this is a good moment for
`/compact` or `/clear`.

### Checkpoint

Run the cell below to see what your history looks like so far. This is roughly
what Elena will see.

In [ ]:
# One line per snapshot, newest first. This is roughly what Elena will read.
run("git log --oneline --decorate")

---

# Phase 3: build your own command (5 minutes)

Elena does not just want features. She wants quality control. Before she
reviews anything, she expects you to have run a QA check.

So make that check a command.

The cell below creates `.claude/commands/qa-check.md` with a starting version.
**Edit it**, in VS Code or right here, so that it reflects how *you* want a QA
report delivered. Bullet points? Severity levels? A pass or fail summary? It is
your command.

Ideas for what it could look for:

- Placeholder text still lurking: lorem ipsum, TODO comments, "click here for
  something".
- Missing alt text on images.
- Placeholder links, `href="#"`.
- Unused CSS rules.
- Accessibility basics: colour contrast, semantic HTML.
- Anything that would make Elena raise an eyebrow.

In [ ]:
QA_CHECK = """Review the current state of index.html and report what is wrong with it.

Check for at least the following:

- Placeholder text that was never replaced: lorem ipsum, TODO comments,
  "click here for something"
- Links whose href is still `#`
- Images with no alt attribute
- CSS rules that nothing on the page uses
- Accessibility basics: heading order, colour contrast, semantic elements

Report every finding as one line, in this shape:

    [severity] location: what is wrong, and the smallest fix

Use three severities: blocker, should-fix, nice-to-have. End with a one line
verdict: is this page ready to show an investor?

Do not change any file. This command only reports.
"""

commands = PROJECT_DIR / ".claude" / "commands"
commands.mkdir(parents=True, exist_ok=True)
qa = commands / "qa-check.md"
if qa.exists():
    # Guarded, because the cell above tells you to EDIT this file. Re-running
    # the notebook must not throw your version away.
    print(qa, "is already there, leaving your version alone.")
else:
    qa.write_text(QA_CHECK, encoding="utf-8")
    print("Wrote", qa)
    print()
    print("Now edit it so the report comes back the way YOU want to read it.")

In [ ]:
run("git add .claude/")
run('git commit -m "chore: add qa-check custom command"')

## Run it

**In your terminal**, inside the Claude Code session:

```
/qa-check
```

A file at `.claude/commands/qa-check.md` becomes the command `/qa-check`, named
after the file. If your Claude Code is old enough to want the namespaced form,
it is `/project:qa-check`.

Custom commands and skills are the same mechanism now: a skill at
`.claude/skills/qa-check/SKILL.md` would give you the same `/qa-check`, plus a
folder for supporting files. See https://code.claude.com/docs/en/slash-commands

Read the output. If Claude found real issues:

> Fix the issues you just found, then save it as a snapshot.

---

# Phase 4: the sunrise review (5 minutes)

Elena is awake. Coffee in hand.

Ask Claude for the whole picture:

> Show me the history of tonight, one line per snapshot.

Then look at it. Does it tell a clean story? Can Elena see what happened
tonight, ticket by ticket, without asking you a single question?

Tag the release:

> Tag the current state as v1.0-investor-demo.

Then open `index.html` one last time. You shipped it.

In [ ]:
run("git log --oneline --decorate")
print()
run("git tag")
print()
run("git status --short")

---

# Bonus challenges, if you have time

**1. Write another command.** Create `.claude/commands/changelog.md`, a command
that asks Claude for a short changelog entry summarising the last feature
added. Test it after your last commit.

**2. Write a skill.** Make a `.claude/skills/brand-voice/SKILL.md` that teaches
Claude how NovaGrid talks: tone of voice, words to avoid, how to write about
sustainability without sounding generic. Then reference it in your next prompt
and see what changes.

**3. The 2 AM curveball.** Mara sends one more message:

> Actually, can we add a dark mode toggle? The lead investor apparently hates
> bright screens.

Ship it. With a clean commit of its own. You know the drill by now.

---

# Take Claude home

You have now seen Claude Code inside a structured workflow. Ways to keep going:

**For your daily work**

- Draft and revise documents, mails and reports from the terminal.
- Set up a `CLAUDE.md` for a personal project and watch the answers change.
- Build commands for the things you repeat: weekly reports, reviews, data
  formatting.

**Put it on the internet**

Your page is still a local file. Deploying it takes minutes:

1. Make a free account at https://vercel.com
2. Install the CLI: `npm install -g vercel`
3. In the project folder, run `vercel` and accept the defaults.

Or ask Claude Code to walk you through it:

> Help me deploy this project to Vercel. Walk me through the setup.

**For a deeper look**

- Compare models on the same task and see whether the difference shows.
- Write increasingly specific `CLAUDE.md` files and watch the output quality
  track it.
- Look at MCP, which connects Claude Code to the tools you already use:
  https://code.claude.com/docs/en/mcp

---

# Reflection, for the group discussion

1. At what point did your `CLAUDE.md` save you from repeating yourself?
2. When did you switch models, and did the output actually change?
3. How did you choose between `/clear` and `/compact`, and what happened to
   Claude's behaviour after each?
4. What would you add to your `CLAUDE.md` if you did this exercise again?
5. Having a snapshot after every ticket, how did that change what you were
   willing to let Claude try?